In [1]:
import pandas as pd
import numpy as np

In [2]:
# --- 1. Data set
np.random.seed(42)
n_musteri = 500

In [3]:
# Özellikler (Features)
kullanim_suresi = np.random.randint(1, 72, n_musteri)
aylik_fatura = np.random.uniform(20.0, 150.0, n_musteri)
mh_arama = np.random.randint(0, 6, n_musteri)

In [4]:
# Şirketi Terk Etme (Churn)
risk_skoru = (mh_arama * 0.8) + (aylik_fatura * 0.02) - (kullanim_suresi * 0.05) - 1.5
olasilik = 1 / (1 + np.exp(-risk_skoru)) # Sigmoid fanc
churn = (olasilik > 0.5).astype(int)

df = pd.DataFrame({
    'Kullanim_Suresi': kullanim_suresi,
    'Aylik_Fatura': aylik_fatura,
    'Musteri_Hizmetleri_Arama': mh_arama,
    'Churn': churn
})

In [5]:
df.head()

,Kullanim_Suresi,Aylik_Fatura,Musteri_Hizmetleri_Arama,Churn
0,52,111.660967,4,1
1,15,52.334140,1,0
2,61,62.932829,4,0
3,21,76.478740,0,0
4,24,52.978683,1,0


In [6]:
df.isnull().sum()

Kullanim_Suresi             0
Aylik_Fatura                0
Musteri_Hizmetleri_Arama    0
Churn                       0
dtype: int64

In [7]:
df['Churn'].value_counts()

Churn
1    295
0    205
Name: count, dtype: int64

In [8]:
df = df.dropna()

In [9]:
X=df.drop('Churn',axis=1)
y=df['Churn']

In [10]:
#traintestsplit
from sklearn.model_selection import train_test_split
X_train,X_test , y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
#pipeline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
pipe= Pipeline (
    [
        ('Standartscaler' ,StandardScaler()),
        ('Logisticregression',LogisticRegression())
    ]
)

In [12]:
pipe.fit(X_train,y_train)

,steps,"[('Standartscaler', ...), ('Logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [13]:
y_pred=pipe.predict(X_test)


In [14]:
y_pred

array([0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0])

In [15]:
param_grid = {
    'Logisticregression__C': [0.01, 0.1, 1, 10, 100],
    'Logisticregression__penalty': ['l1', 'l2']
}

In [16]:
#GridSearchCV
from sklearn.model_selection import GridSearchCV
grid_search=GridSearchCV(scoring='accuracy',estimator=pipe,param_grid=param_grid ,cv=5)


In [17]:
grid_search.fit(X_train,y_train)

C:\Users\Monster\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
25 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Monster\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Monster\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\loca

,estimator,Pipeline(step...egression())])
,param_grid,"{'Logisticregression__C': [0.01, 0.1, ...], 'Logisticregression__penalty': ['l1', 'l2']}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [18]:
grid_search.best_params_

{'Logisticregression__C': 10, 'Logisticregression__penalty': 'l2'}

In [26]:
grid_search.best_score_

np.float64(0.9974999999999999)

In [27]:
grid_search.best_index_

np.int64(7)

In [19]:
y_pred=grid_search.predict(X_test)

In [20]:
y_pred

array([0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0])

In [21]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
score=accuracy_score(y_pred,y_test)

In [22]:
score

1.0

In [23]:
print(confusion_matrix(y_pred,y_test))
print(classification_report(y_pred,y_test))

[[40  0]
 [ 0 60]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        40
           1       1.00      1.00      1.00        60

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

